In [36]:
import json
import re
import ast

pattern = r"\[.*\]"
with open("ollamaF1Scorer.ipynb") as file:
    jsonContent = json.load(file)
phi4List = []
for i in jsonContent['cells']:
    a = [j for j in i['source'] if "chat" in j][0]
    b = ast.literal_eval(re.search(pattern, a)[0])
    phi4List.append(b)
print(phi4List)

[['challenging', 'uncooperative behavior', 'noncompliant', 'frequent episodes of cursing at healthcare staff', 'difficult patient', 'refused several exams', 'resistant to modifying his diet', 'poor dietary habits', 'idiotic short stature'], ['noncompliant', 'nonadherent', 'uncooperative', 'refusing exams and medication administration', 'compliance issues'], ['challenging', 'difficult to deal with', 'noncompliant', 'refusing to answer our questions directly', 'contradicting herself', 'making the assessment more complicated than necessary', 'resistant during the exam', 'inconsistent in narration', 'exhibited compliance issues throughout her stay', 'refusing medications and declining exams', 'noncompliant behavior', 'delayed discharge', 'difficult'], ['non-compliant', 'frequently missed follow-up appointments', 'medication adherence issues', 'frustration about medical advice provided', 'showing a pattern of medication adherence issues', 'alcohol over the recommended guidelines', 'social g

In [ ]:
import pandas as pd
from langchain_ollama import OllamaLLM

model = OllamaLLM(model="llama3.1")

def askOllama(prompt):
    result = model.invoke(input=prompt)
    return result

df = pd.read_csv("/Users/sagewong/git/StigmatizingLanguageProject/FinalFinalAnnnotatedData.csv")
allList = []
for index, i in enumerate(phi4List):
    clinicalNote = df.iloc[index]["Completion"]
    clinicalNote = re.sub(r'^.*?\*\*History of Present Illness:\*\*', '', clinicalNote, flags=re.DOTALL)
    sentences = clinicalNote.split("**")
    sentences = [item for part in sentences for item in part.split("-")]
    for word in i:
        if len([j for j in sentences if word in j]) > 0:
            text = [j for j in sentences if word in j][0]
            allList.append([word, text])

def group_by_second_index(data):
    result = {}

    for element in data:
        key = element[1]  # The second index (the grouping key)
        value = element[0]  # The first index (the value for the key)

        if key in result:
            result[key].append(value)  # If the key exists, append the value to the list
        else:
            result[key] = [value]  # If the key doesn't exist, create a new list with the value

    return result

z = group_by_second_index(allList)

for key, value in z.items():
    print(key.replace("\n", ""))
    print(value)

    prompt = "You are a professional linguist whose job is to replace stigmatizing language in clinical notes. If you see labels such as diabetic or abuser, replace these labels with person first language such as \"person who has diabetes\" or\"person with a substance abuse disorder\". If you see words like challenging or uncooperative, replace them with more respectful alternatives. Here is the sentence: " + str(key) + " And here is are the stigmatizing words you must replace: " + str(value) + ". Return to me only the corrected sentence"
    print(askOllama(prompt))
    print("---------")



 Mohammad, a challenging patient with a history of uncooperative behavior, presents with concerns about his significantly short stature compared to peers. Despite recommendations for nutritional interventions, Mohammad has been noncompliant, citing his preferences for junk food. His mother reports frequent cursing at nurses during previous medical consultations, further complicating collaborative treatment efforts.
 Throughout his stay, Mohammad was a difficult patient, reflecting a pattern of challenging interactions and unreliable historical recounting. Compliance issues were evident, as he refused several exams and was resistant to modifying his diet, despite clear evidence linking his nutritional habits to his short stature. Cursing at staff persisted, complicating our attempts to provide care.
 Nutritional assessment indicates poor dietary habits
Evaluation focused on Noel's short stature, which included blood tests and consultations with endocrinology. Noel was often uncooperativ